In [2]:
from typing import Tuple, Dict, Any

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import (
    CLIPVisionModel,
    CLIPTextModel,
    CLIPTokenizer,
    CLIPImageProcessor,
    CLIPModel,
)
from transformers.modeling_outputs import BaseModelOutput, ModelOutput
import inspect

In [3]:
MODEL_NAME = "openai/clip-vit-base-patch32"

In [4]:
torch.manual_seed(0)

In [5]:
clip_vision_model = CLIPVisionModel.from_pretrained(MODEL_NAME)
clip_text_model = CLIPTextModel.from_pretrained(MODEL_NAME)
tokenizer = CLIPTokenizer.from_pretrained(MODEL_NAME)
processor = CLIPImageProcessor.from_pretrained(MODEL_NAME)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_at

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

[transformers] CLIPTextModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
vision_model.encoder.l

In [6]:
clip_vision_model

CLIPVisionModel(
  (embeddings): CLIPVisionEmbeddings(
    (patch_embedding): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
    (position_embedding): Embedding(50, 768)
  )
  (pre_layrnorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (encoder): CLIPEncoder(
    (layers): ModuleList(
      (0-11): 12 x CLIPEncoderLayer(
        (self_attn): CLIPAttention(
          (k_proj): Linear(in_features=768, out_features=768, bias=True)
          (v_proj): Linear(in_features=768, out_features=768, bias=True)
          (q_proj): Linear(in_features=768, out_features=768, bias=True)
          (out_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): CLIPMLP(
          (activation_fn): QuickGELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
        )
    

In [7]:
clip_vision_model.encoder

CLIPEncoder(
  (layers): ModuleList(
    (0-11): 12 x CLIPEncoderLayer(
      (self_attn): CLIPAttention(
        (k_proj): Linear(in_features=768, out_features=768, bias=True)
        (v_proj): Linear(in_features=768, out_features=768, bias=True)
        (q_proj): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
      )
      (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): CLIPMLP(
        (activation_fn): QuickGELUActivation()
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (fc2): Linear(in_features=3072, out_features=768, bias=True)
      )
      (layer_norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    )
  )
)

In [8]:
print(inspect.getsource(clip_vision_model.encoder.forward))

    def forward(
        self,
        inputs_embeds,
        attention_mask: torch.Tensor | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> BaseModelOutput:
        hidden_states = inputs_embeds
        for encoder_layer in self.layers:
            hidden_states = encoder_layer(
                hidden_states,
                attention_mask,
                **kwargs,
            )

        return BaseModelOutput(
            last_hidden_state=hidden_states,
        )



In [9]:
print(inspect.getsource(clip_vision_model.forward))

    @merge_with_config_defaults
    @capture_outputs(tie_last_hidden_states=False)
    @auto_docstring
    def forward(
        self,
        pixel_values: torch.FloatTensor | None = None,
        interpolate_pos_encoding: bool | None = False,
        **kwargs: Unpack[TransformersKwargs],
    ) -> BaseModelOutputWithPooling:
        r"""
        Example:

        ```python
        >>> from PIL import Image
        >>> import httpx
        >>> from io import BytesIO
        >>> from transformers import AutoProcessor, CLIPVisionModel

        >>> model = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch32")
        >>> processor = AutoProcessor.from_pretrained("openai/clip-vit-base-patch32")

        >>> url = "http://images.cocodataset.org/val2017/000000039769.jpg"
        >>> with httpx.stream("GET", url) as response:
        ...     image = Image.open(BytesIO(response.read()))

        >>> inputs = processor(images=image, return_tensors="pt")

        >>> outputs = model(**i

In [10]:
class UniResImageEncoder(nn.Module):
    def __init__(self, image_encoder: CLIPVisionModel) -> None:
        super().__init__()
        self.image_encoder = image_encoder
        d_model = self.image_encoder.config.hidden_size  # 768
        self.low_tokens = nn.Parameter(torch.rand(64, d_model))
        self.high_tokens = nn.Parameter(torch.rand(8, d_model))

    def forward(self, pixel_values: torch.Tensor) -> Dict[str, torch.Tensor]:
        # tokenize the image + positional embedding
        # (B,50,768)
        hidden_states = self.image_encoder.embeddings(pixel_values)
        batch_size, seq_len, _ = hidden_states.shape

        # insert low level group tokens
        # (B,64,768)
        low_tokens = self.low_tokens.unsqueeze(0).expand(batch_size, -1, -1)
        # (B,114,768)
        hidden_states = torch.cat((hidden_states, low_tokens), dim=1)

        # norm
        hidden_states = self.image_encoder.pre_layrnorm(hidden_states)

        # first encoder half
        for encoder_layer in self.image_encoder.encoder.layers[:6]:
            hidden_states = encoder_layer(hidden_states, attention_mask=None)

        # take out low level group tokens
        low_tokens = hidden_states[:, seq_len:, :]
        # (B,50,768)
        hidden_states = hidden_states[:, :seq_len, :]

        # insert high level group tokens
        # (B,8,768)
        high_tokens = self.high_tokens.unsqueeze(0).expand(batch_size, -1, -1)
        # (B,58,768)
        hidden_states = torch.cat((hidden_states, high_tokens), dim=1)

        # second encoder half
        for encoder_layer in self.image_encoder.encoder.layers[6:]:
            hidden_states = encoder_layer(hidden_states, attention_mask=None)

        # take out high level group tokens
        high_tokens = hidden_states[:, seq_len:, :]
        # (B,50,768)
        last_hidden_state = hidden_states[:, :seq_len, :]

        # CLS token
        # (B,768)
        pooled_output = last_hidden_state[:, 0, :]
        # (B,768)
        pooled_output = self.image_encoder.post_layernorm(pooled_output)

        return {
            "last_hidden_state": last_hidden_state,
            "pooler_output": pooled_output,
            "low_tokens": low_tokens,
            "high_tokens": high_tokens,
        }

In [36]:
class LanguageGuidedRegionFilter(nn.Module):
    def __init__(self, embed_dim, num_heads, ffn_hidden_dim) -> None:
        super().__init__()

        self.cross_attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)

        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ffn_hidden_dim),
            nn.ReLU(),
            nn.Linear(ffn_hidden_dim, embed_dim),
        )

        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(
        self, group_feature: torch.Tensor, text_feature: torch.Tensor
    ) -> torch.Tensor:
        attn_output, _ = self.cross_attn(
            query=group_feature, key=text_feature, value=text_feature
        )

        norm1_output = self.norm1(group_feature + attn_output)
        ffn_output = self.ffn(norm1_output)
        norm2_output = self.norm2(norm1_output + ffn_output)

        return norm2_output


In [12]:
class UniRes(nn.Module):
    def __init__(
        self, image_encoder: CLIPVisionModel, text_encoder: CLIPTextModel
    ) -> None:
        super().__init__()
        self.image_encoder = UniResImageEncoder(image_encoder)
        self.text_encoder = text_encoder
        self.image_projection = nn.Linear(768, 512)
        self.text_projection = nn.Linear(512, 512)
        self.lrf = LanguageGuidedRegionFilter(
            embed_dim=512, num_heads=8, ffn_hidden_dim=512 * 4
        )

    def forward(
        self,
        pixel_values: torch.Tensor,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:
        text_encoder_output = self.text_encoder(input_ids, attention_mask)
        text_feature = text_encoder_output["last_hidden_state"]

        image_encoder_output = self.image_encoder(pixel_values)
        image_feature = image_encoder_output["last_hidden_state"]
        low_tokens = image_encoder_output["low_tokens"]
        high_tokens = image_encoder_output["high_tokens"]


In [13]:
clip_inputs = tokenizer(
    ["i am subaru", "as the strongest curse jogoat"],
    padding=True,
    truncation=True,
    return_tensors="pt",
)
clip_outputs = clip_text_model(**clip_inputs)
clip_outputs

BaseModelOutputWithPooling(last_hidden_state=tensor([[[ 3.3929e-01,  1.1646e-01,  1.0195e-01,  ...,  2.4677e-01,
           5.9064e-01,  1.0130e-01],
         [ 6.9760e-01, -8.3717e-01,  4.3726e-01,  ...,  1.1181e+00,
          -1.4406e-01, -2.9393e-01],
         [-3.7932e-01,  9.1991e-01,  4.3670e-02,  ...,  2.2799e+00,
          -4.8892e-01, -1.1454e+00],
         ...,
         [ 1.1276e+00,  3.9555e-01, -2.3678e-01,  ...,  2.4398e+00,
          -2.3462e-01, -5.6630e-01],
         [ 1.2047e+00,  3.5047e-01, -2.7333e-01,  ...,  2.4482e+00,
          -1.4395e-01, -5.1987e-01],
         [ 1.2211e+00,  3.5429e-01, -2.4436e-01,  ...,  2.4187e+00,
          -1.1318e-01, -4.9294e-01]],

        [[ 3.3929e-01,  1.1646e-01,  1.0195e-01,  ...,  2.4677e-01,
           5.9064e-01,  1.0130e-01],
         [ 8.2823e-01, -1.2106e+00,  6.5737e-02,  ...,  1.4178e+00,
           7.5970e-01,  1.0249e-01],
         [-3.7181e-02, -7.1953e-01,  7.8511e-02,  ...,  2.4186e-01,
          -1.1970e-03, -5.1704e

In [14]:
clip_outputs.pooler_output.shape

torch.Size([2, 512])

In [15]:
from PIL import Image
import httpx
from io import BytesIO

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
with httpx.stream("GET", url) as response:
    image = Image.open(BytesIO(response.read()))

image_inputs = processor(images=[image, image, image], return_tensors="pt")

image_outputs = clip_vision_model(**image_inputs)

In [16]:
yo = clip_vision_model.embeddings(image_inputs["pixel_values"])

In [17]:
yo.shape

torch.Size([3, 50, 768])

In [18]:
jogoat = nn.Parameter(torch.rand(8, 768))
jogoat.shape

torch.Size([8, 768])

In [19]:
goat = jogoat.unsqueeze(0).expand(3, -1, -1)
goat.shape

torch.Size([3, 8, 768])

In [20]:
torch.cat((yo, goat), dim=1).shape

torch.Size([3, 58, 768])

In [21]:
image_outputs.pooler_output.shape

torch.Size([3, 768])

In [22]:
clip_text_model

CLIPTextModel(
  (embeddings): CLIPTextEmbeddings(
    (token_embedding): Embedding(49408, 512)
    (position_embedding): Embedding(77, 512)
  )
  (encoder): CLIPEncoder(
    (layers): ModuleList(
      (0-11): 12 x CLIPEncoderLayer(
        (self_attn): CLIPAttention(
          (k_proj): Linear(in_features=512, out_features=512, bias=True)
          (v_proj): Linear(in_features=512, out_features=512, bias=True)
          (q_proj): Linear(in_features=512, out_features=512, bias=True)
          (out_proj): Linear(in_features=512, out_features=512, bias=True)
        )
        (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (mlp): CLIPMLP(
          (activation_fn): QuickGELUActivation()
          (fc1): Linear(in_features=512, out_features=2048, bias=True)
          (fc2): Linear(in_features=2048, out_features=512, bias=True)
        )
        (layer_norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      )
    )
  )
  (final_layer_norm): LayerNo

In [23]:
clip_vision_model

CLIPVisionModel(
  (embeddings): CLIPVisionEmbeddings(
    (patch_embedding): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
    (position_embedding): Embedding(50, 768)
  )
  (pre_layrnorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (encoder): CLIPEncoder(
    (layers): ModuleList(
      (0-11): 12 x CLIPEncoderLayer(
        (self_attn): CLIPAttention(
          (k_proj): Linear(in_features=768, out_features=768, bias=True)
          (v_proj): Linear(in_features=768, out_features=768, bias=True)
          (q_proj): Linear(in_features=768, out_features=768, bias=True)
          (out_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): CLIPMLP(
          (activation_fn): QuickGELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
        )
    

In [24]:
import torch
from transformers import AutoProcessor, CLIPModel
from transformers.image_utils import load_image

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = AutoProcessor.from_pretrained("openai/clip-vit-base-patch32")

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = load_image(url)

clip_inputs = processor(
    text=["a photo of a cat", "a photo of a dog"],
    images=image,
    return_tensors="pt",
    padding=True,
)

with torch.inference_mode():
    clip_outputs = model(**clip_inputs)
logits_per_image = (
    clip_outputs.logits_per_image
)  # this is the image-text similarity score
probs = logits_per_image.softmax(
    dim=1
)  # we can take the softmax to get the label probabilities

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [25]:
for key in clip_outputs.keys():
    print(key)

logits_per_image
logits_per_text
text_embeds
image_embeds
text_model_output
vision_model_output


In [26]:
clip_outputs.image_embeds.shape

torch.Size([1, 512])

In [27]:
isinstance(clip_outputs, ModelOutput)

True

In [28]:
model

CLIPModel(
  (text_model): CLIPTextModel(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05, eleme

In [29]:
clip_vision_model

CLIPVisionModel(
  (embeddings): CLIPVisionEmbeddings(
    (patch_embedding): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
    (position_embedding): Embedding(50, 768)
  )
  (pre_layrnorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (encoder): CLIPEncoder(
    (layers): ModuleList(
      (0-11): 12 x CLIPEncoderLayer(
        (self_attn): CLIPAttention(
          (k_proj): Linear(in_features=768, out_features=768, bias=True)
          (v_proj): Linear(in_features=768, out_features=768, bias=True)
          (q_proj): Linear(in_features=768, out_features=768, bias=True)
          (out_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): CLIPMLP(
          (activation_fn): QuickGELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
        )
    

In [30]:
uni_image_encoder = UniResImageEncoder(clip_vision_model)

In [31]:
uni_image_encoder

UniResImageEncoder(
  (image_encoder): CLIPVisionModel(
    (embeddings): CLIPVisionEmbeddings(
      (patch_embedding): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
      (position_embedding): Embedding(50, 768)
    )
    (pre_layrnorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
          

In [32]:
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
with httpx.stream("GET", url) as response:
    image = Image.open(BytesIO(response.read()))

image_inputs = processor(images=[image, image, image], return_tensors="pt")

In [33]:
pixel_values = image_inputs["pixel_values"]
outputs = uni_image_encoder(pixel_values)

In [34]:
for k, v in outputs.items():
    print(f"{k}: {v.shape}")

last_hidden_state: torch.Size([3, 50, 768])
pooler_output: torch.Size([3, 768])
low_tokens: torch.Size([3, 64, 768])
high_tokens: torch.Size([3, 8, 768])


In [35]:
image_inputs["pixel_values"].shape

torch.Size([3, 3, 224, 224])

In [39]:
lrf = LanguageGuidedRegionFilter(512, 8, 512 * 4)
group_tensor = torch.rand(16, 8, 512)
text_tensor = torch.rand(16, 30, 512)
lrf(group_tensor, text_tensor).shape

torch.Size([16, 8, 512])